# Mixed fixed-point `.mem` conversion

This notebook converts the uploaded weight and bias `.mem` files into a **single common fixed-point format**.

It uses these segment definitions:

- **Weights**
  - first 6272 rows: layer 1 weights, 9 fractional bits
  - next 240 rows: layer 2 weights, 7 fractional bits
  - next 60 rows: layer 3 weights, 6 fractional bits
- **Biases**
  - first 8 rows: layer 1 biases, 8 fractional bits
  - next 6 rows: layer 2 biases, 7 fractional bits
  - next 2 rows: layer 3 biases, 8 fractional bits

Default target format: **common fractional bits = 9**.


In [ ]:

from pathlib import Path
import math

# ============================================================
# Mixed fixed-point .mem conversion utility
# ------------------------------------------------------------
# Assumptions from your architecture:
#   - Input files are 8-bit two's-complement binary strings
#   - Different contiguous sections use different fractional lengths
#   - We convert everything to ONE common fractional length
#
# Default choice here:
#   COMMON_FRAC_BITS = 9
# This preserves all of your original values exactly, because 9 is the
# largest fractional-bit count used anywhere in your weights/biases.
# ============================================================

# -------- user-editable settings --------
INPUT_DIR = Path(".")          # folder containing W_PE*.mem and B_PE*.mem
OUTPUT_DIR = INPUT_DIR / "converted_q9"
COMMON_FRAC_BITS = 9

# Set to None to auto-compute minimum required width.
# Set to 16 if you prefer all outputs to be 16-bit for convenience.
OUTPUT_WIDTH = 16
# OUTPUT_WIDTH = None

WEIGHT_FILES = [f"W_PE{i}.mem" for i in range(1, 6)]
BIAS_FILES   = [f"B_PE{i}.mem" for i in range(1, 6)]

# Contiguous segment definitions: (segment_name, number_of_entries, frac_bits)
WEIGHT_SEGMENTS = [
    ("layer1_weights", 6272, 9),
    ("layer2_weights",  240, 7),
    ("layer3_weights",   60, 6),
]

BIAS_SEGMENTS = [
    ("layer1_biases", 8, 8),
    ("layer2_biases", 6, 7),
    ("layer3_biases", 2, 8),
]


def twos_comp_to_int(bitstr: str) -> int:
    """Interpret a binary string as a signed two's-complement integer."""
    width = len(bitstr)
    val = int(bitstr, 2)
    if val >= (1 << (width - 1)):
        val -= (1 << width)
    return val


def int_to_twos_comp(val: int, width: int) -> str:
    """Return signed integer val as width-bit two's-complement binary string."""
    min_val = -(1 << (width - 1))
    max_val =  (1 << (width - 1)) - 1
    if not (min_val <= val <= max_val):
        raise OverflowError(
            f"value {val} does not fit in signed {width}-bit range "
            f"[{min_val}, {max_val}]"
        )
    if val < 0:
        val = (1 << width) + val
    return format(val, f"0{width}b")


def real_from_raw(raw_int: int, frac_bits: int) -> float:
    return raw_int / (2 ** frac_bits)


def convert_raw_int(raw_int: int, src_frac_bits: int, dst_frac_bits: int) -> int:
    """
    Convert the *stored integer* from Q?.src_frac_bits to Q?.dst_frac_bits
    while preserving the represented real value.
    """
    shift = dst_frac_bits - src_frac_bits
    if shift >= 0:
        return raw_int << shift
    else:
        # Right shift would only be needed if dst_frac_bits < src_frac_bits.
        # Arithmetic shift preserves sign for Python ints.
        return raw_int >> (-shift)


def read_mem_lines(path: Path):
    with open(path, "r") as f:
        return [line.strip() for line in f if line.strip()]


def write_mem_lines(path: Path, lines):
    with open(path, "w") as f:
        for line in lines:
            f.write(line + "\n")


def validate_segments(total_len: int, segments, label: str):
    seg_sum = sum(count for _, count, _ in segments)
    if total_len != seg_sum:
        raise ValueError(
            f"{label}: file has {total_len} rows, but segment definition sums to {seg_sum}"
        )


def collect_converted_ints(bit_lines, segments, common_frac_bits):
    """
    Returns a list of dicts with detailed conversion info for each row.
    """
    rows = []
    idx = 0
    for seg_name, seg_len, seg_frac in segments:
        for local_idx in range(seg_len):
            global_idx = idx + local_idx
            raw_bits = bit_lines[global_idx]
            raw_int = twos_comp_to_int(raw_bits)
            converted_int = convert_raw_int(raw_int, seg_frac, common_frac_bits)
            rows.append({
                "segment": seg_name,
                "index_in_file": global_idx,
                "index_in_segment": local_idx,
                "src_bits": raw_bits,
                "src_int": raw_int,
                "src_frac_bits": seg_frac,
                "real_value": real_from_raw(raw_int, seg_frac),
                "dst_frac_bits": common_frac_bits,
                "dst_int": converted_int,
            })
        idx += seg_len
    return rows


def min_signed_width_for_values(values):
    max_abs = max(abs(v) for v in values) if values else 0
    width = 1
    while max_abs >= (1 << (width - 1)):
        width += 1
    return max(width, 2)


def convert_file(
    input_path: Path,
    output_path: Path,
    segments,
    common_frac_bits=9,
    output_width=None,
    preview_rows=6,
):
    bit_lines = read_mem_lines(input_path)
    validate_segments(len(bit_lines), segments, input_path.name)

    rows = collect_converted_ints(bit_lines, segments, common_frac_bits)
    converted_ints = [r["dst_int"] for r in rows]

    if output_width is None:
        output_width = min_signed_width_for_values(converted_ints)

    output_lines = [int_to_twos_comp(v, output_width) for v in converted_ints]
    write_mem_lines(output_path, output_lines)

    print(f"\nConverted {input_path.name} -> {output_path.name}")
    print(f"  rows              : {len(bit_lines)}")
    print(f"  common frac bits  : {common_frac_bits}")
    print(f"  output width      : {output_width} bits")

    print("  preview:")
    for r, out_bits in zip(rows[:preview_rows], output_lines[:preview_rows]):
        print(
            f"    idx={r['index_in_file']:4d}  "
            f"{r['segment']:15s}  "
            f"src={r['src_bits']} ({r['src_int']:4d}, Q* .{r['src_frac_bits']})  "
            f"-> dst={out_bits} ({r['dst_int']:5d}, Q* .{common_frac_bits})  "
            f"real={r['real_value']:+.8f}"
        )

    return rows, output_width


def convert_all(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    common_frac_bits=COMMON_FRAC_BITS,
    output_width=OUTPUT_WIDTH,
):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    all_widths = []

    print("=" * 72)
    print("Converting WEIGHT files")
    print("=" * 72)
    for name in WEIGHT_FILES:
        rows, used_width = convert_file(
            input_dir / name,
            output_dir / name,
            WEIGHT_SEGMENTS,
            common_frac_bits=common_frac_bits,
            output_width=output_width,
        )
        all_widths.append((name, used_width))

    print("\n" + "=" * 72)
    print("Converting BIAS files")
    print("=" * 72)
    for name in BIAS_FILES:
        rows, used_width = convert_file(
            input_dir / name,
            output_dir / name,
            BIAS_SEGMENTS,
            common_frac_bits=common_frac_bits,
            output_width=output_width,
        )
        all_widths.append((name, used_width))

    print("\nDone.")
    print("Output folder:", output_dir.resolve())
    print("Widths used:")
    for name, w in all_widths:
        print(f"  {name}: {w} bits")


# -------------------------------------------------------------------
# Optional helper: inspect any single entry by index and segment map
# -------------------------------------------------------------------
def inspect_entry(input_path, segments, idx, common_frac_bits=COMMON_FRAC_BITS):
    input_path = Path(input_path)
    lines = read_mem_lines(input_path)
    validate_segments(len(lines), segments, input_path.name)

    rows = collect_converted_ints(lines, segments, common_frac_bits)
    r = rows[idx]
    print(f"file           : {input_path.name}")
    print(f"index_in_file  : {r['index_in_file']}")
    print(f"segment        : {r['segment']}")
    print(f"src_bits       : {r['src_bits']}")
    print(f"src_int        : {r['src_int']}")
    print(f"src_frac_bits  : {r['src_frac_bits']}")
    print(f"real_value     : {r['real_value']}")
    print(f"dst_int (Q*.{common_frac_bits}) : {r['dst_int']}")


# =========================
# Execute the conversion
# =========================
convert_all()
